

> This notebook contains the code used for running the translations from our dataset CodeNet_Final. We will use this structure for accessing the COBOL files to translate: **CodeNet_Final/PID/COBOL/abc.cbl or .cob**

> There are 3 translators used - COBOL4J, XMainframe-7B and Qwen-7B. The code to install any dependencies and run the translation for each of these is provided below.










# COBOL4J





> Installing COBOL4J dependencies


In [ ]:
import os
import subprocess

# ===== INSTALL OPENSOURCECOBOL4J =====

print("="*80)
print("INSTALLING OPENSOURCECOBOL4J")
print("="*80)

# Install dependencies
print("\n Installing dependencies...")
os.system("apt-get update -qq")
os.system("apt-get install -y -qq default-jdk build-essential bison flex gettext texinfo libgmp-dev autoconf")

# Download and install opensourcecobol4j
print("\n Downloading opensourcecobol4j v1.1.16...")
os.system("curl -L -o opensourcecobol4j-v1.1.16.tar.gz https://github.com/opensourcecobol/opensourcecobol4j/archive/refs/tags/v1.1.16.tar.gz")

print("\n Extracting...")
os.system("tar zxf opensourcecobol4j-v1.1.16.tar.gz")

print("\n Configuring and building...")
os.chdir("opensourcecobol4j-1.1.16")
os.system("./configure --prefix=/usr/")
os.system("make")
os.system("make install")

# Set CLASSPATH
print("\n Setting CLASSPATH...")
os.environ['CLASSPATH'] = '/usr/lib/opensourcecobol4j/libcobj.jar'

# Return to original directory
os.chdir("/content")

print("\n OpenSourceCOBOL4J installed successfully!")

# Verify installation
result = subprocess.run(['cobj', '--version'], capture_output=True, text=True)
print(f"\n{result.stdout}")






> Testing compilation with the libcobj.jar works - create a sample java file and run it with the expected output (here we used a simple SortNum test)





In [ ]:

import os
import subprocess

# Paths
LIBCOBJ_JAR = '/usr/lib/opensourcecobol4j/libcobj.jar'
JAVA_FILE = 'SORTNUM.java' # Add some test file
INPUT_FILE = 'input1.txt' # Add some test input

print("="*80)
print("COMPILING AND RUNNING TRANSLATED JAVA")
print("="*80)

# Verify files
if not os.path.exists(LIBCOBJ_JAR):
    print(f" Error: {LIBCOBJ_JAR} not found!")
    exit(1)

if not os.path.exists(JAVA_FILE):
    print(f" Error: {JAVA_FILE} not found!")
    exit(1)

print(f" Found libcobj.jar: {LIBCOBJ_JAR}")
print(f" Found Java file: {JAVA_FILE}")

# Read and show input
if os.path.exists(INPUT_FILE):
    print(f" Found input file: {INPUT_FILE}")
    with open(INPUT_FILE, 'r') as f:
        input_data = f.read()
    print("\n Input contents:")
    print("-" * 40)
    print(input_data)
    print("-" * 40)
else:
    print(f" No input file found")
    input_data = None

print("\n" + "="*80)
print("COMPILING")
print("="*80)

# Compile
compile_result = subprocess.run(
    ['javac', '-cp', LIBCOBJ_JAR, JAVA_FILE],
    capture_output=True,
    text=True
)

if compile_result.returncode != 0:
    print(" Compilation failed!")
    print("\nSTDERR:")
    print(compile_result.stderr)
    exit(1)

print(" Compilation successful!")

print("\n" + "="*80)
print("RUNNING PROGRAM")
print("="*80)

# Run with proper output capture
run_result = subprocess.run(
    ['java', '-cp', f'.:{LIBCOBJ_JAR}', 'SORTNUM'],
    input=input_data,
    capture_output=True,
    text=True,
    timeout=10
)

print("\n PROGRAM OUTPUT:")
print("=" * 80)
if run_result.stdout:
    print(run_result.stdout)
else:
    print("(no output)")

if run_result.stderr:
    print("\n STDERR:")
    print(run_result.stderr)

print("=" * 80)
print(f"\n Exit code: {run_result.returncode}")

# Explain what the program should do
print("\n" + "="*80)
print("EXPECTED BEHAVIOR")
print("="*80)
print("""
This COBOL program:
1. Reads N (number of pairs)
2. For each pair of digits:
   - If both digits are the same, increment counter
   - If counter reaches 3, print "Yes" and exit
   - If digits differ, reset counter to 0
3. If we finish without 3 consecutive matches, print "No"

Your input: 5 pairs
Pair 1: 1 2 (different) → counter = 0
Pair 2: 6 6 (same) → counter = 1
Pair 3: 4 4 (same) → counter = 2
Pair 4: 3 3 (same) → counter = 3 → should print "Yes"
Pair 5: 2 2 (not checked, already exited)

Expected output: "Yes"
""")



>Testing COBOL4J translation works - use a sample COBOL file and examine the resulting Java file



In [ ]:
#TEST TRANSLATION
import os
import subprocess


print("\n" + "="*80)
print("TESTING TRANSLATION")
print("="*80)

COBOL_FILE = 'test.cob'  # Create a test COBOL file

if not os.path.exists(COBOL_FILE):
    print(f"\n Error: {COBOL_FILE} not found!")
    print(f"   Please upload your COBOL file to /content/")
else:
    print(f"\n Reading {COBOL_FILE}...")

    with open(COBOL_FILE, 'r', encoding='utf-8') as f:
        cobol_code = f.read()

    print(f"   Original file size: {len(cobol_code)} characters")

    # Fix COBOL formatting - ensure proper column layout
    print("\n🔧 Fixing COBOL format (columns 7-72)...")
    lines = cobol_code.split('\n')
    fixed_lines = []

    for line in lines:
        # Remove any leading/trailing whitespace
        line = line.rstrip()

        # Skip empty lines
        if not line.strip():
            fixed_lines.append('')
            continue

        # COBOL format: columns 1-6 are sequence, 7 is indicator, 8-72 is code
        # For free-format COBOL, we'll just ensure proper spacing
        # Add 6 spaces for sequence area, then the line content starting at column 8
        fixed_line = '      ' + ' ' + line.lstrip()
        fixed_lines.append(fixed_line)

    fixed_cobol = '\n'.join(fixed_lines)

    # Save the fixed version
    fixed_file = COBOL_FILE.replace('.cob', '_fixed.cob')
    with open(fixed_file, 'w', encoding='utf-8') as f:
        f.write(fixed_cobol)

    print(f"   Fixed COBOL saved as: {fixed_file}")
    print(f"   First 200 chars of fixed code:\n{fixed_cobol[:200]}...")

    print(f"\n Running OpenSourceCOBOL4J translation on fixed file...")

    # Run cobj command on the FIXED file
    result = subprocess.run(
        ['cobj', fixed_file],
        capture_output=True,
        text=True,
        timeout=60
    )

    # Get base name (should be SORTNUM based on PROGRAM-ID)
    # Try to extract PROGRAM-ID from the file
    program_id = None
    for line in fixed_lines:
        if 'PROGRAM-ID' in line.upper():
            parts = line.split('.')
            if len(parts) >= 2:
                program_id = parts[1].strip().split()[0]
                break

    if not program_id:
        program_id = fixed_file.replace('_fixed.cob', '').replace('.cob', '')

    java_file = f'{program_id}.java'

    # Check if Java file was generated
    if os.path.exists(java_file):
        print(" Translation successful!\n")

        # Read and display the Java code
        with open(java_file, 'r', encoding='utf-8') as f:
            java_code = f.read()

        print("="*80)
        print("GENERATED JAVA CODE:")
        print("="*80)
        print(java_code)
        print("="*80)

        print(f"\n Java code saved as: {java_file}")
        print(f" Generated file size: {len(java_code)} characters")

        # Also check if .class file was created
        class_file = f'{program_id}.class'
        if os.path.exists(class_file):
            print(f" Compiled class file: {class_file}")

    else:
        print(" Translation failed!\n")
        print("STDOUT:")
        print(result.stdout)
        print("\nSTDERR:")
        print(result.stderr)
        print(f"\nReturn code: {result.returncode}")

print("\n Done!")

COBOL4J Translations for MultiModule programs

In [ ]:
import os
import json
import subprocess
import shutil
from tqdm import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

SOURCE_DIR = 'CodeNet_Dataset'
OUTPUT_DIR = 'cobol4j_results'
TEMP_DIR   = 'temp_cobol_rerun'

PIDS_TO_RERUN = ['p03275', 'p03290', 'p03297']

os.makedirs(TEMP_DIR, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================

def clean_temp_dir():
    for f in os.listdir(TEMP_DIR):
        fpath = os.path.join(TEMP_DIR, f)
        try:
            if os.path.isfile(fpath):
                os.remove(fpath)
            elif os.path.isdir(fpath):
                shutil.rmtree(fpath)
        except Exception as e:
            print(f"  [WARN] Could not remove {fpath}: {e}")

def fix_cobol_format(cobol_code):
    lines = cobol_code.split('\n')
    fixed_lines = []
    for line in lines:
        line = line.rstrip()
        if not line.strip():
            fixed_lines.append('')
            continue
        fixed_lines.append('      ' + ' ' + line.lstrip())
    return '\n'.join(fixed_lines)

def translate_cobol_to_java(cobol_path):
    try:
        clean_temp_dir()

        with open(cobol_path, 'r', encoding='utf-8', errors='ignore') as f:
            cobol_code = f.read()

        fixed_cobol = fix_cobol_format(cobol_code)

        temp_cobol_file = os.path.join(TEMP_DIR, 'input.cob')
        with open(temp_cobol_file, 'w', encoding='utf-8') as f:
            f.write(fixed_cobol)

        result = subprocess.run(
            ['cobj', temp_cobol_file],
            capture_output=True, text=True,
            timeout=60, cwd=TEMP_DIR
        )

        # Get ALL java files generated
        java_files = [f for f in os.listdir(TEMP_DIR) if f.endswith('.java')]

        if not java_files:
            return {}, result.stderr or "No Java file generated"

        # Read all of them
        outputs = {}
        for jf in java_files:
            with open(os.path.join(TEMP_DIR, jf), 'r', encoding='utf-8') as f:
                outputs[jf] = f.read()

        return outputs, None

    except subprocess.TimeoutExpired:
        return {}, "Translation timeout (>60s)"
    except Exception as e:
        return {}, str(e)
    finally:
        clean_temp_dir()

# ============================================================
# MAIN
# ============================================================

print("=" * 60)
print(f"Rerunning translation for: {PIDS_TO_RERUN}")
print("=" * 60)

for pid in PIDS_TO_RERUN:
    print(f"\n>> Processing {pid}...")

    cobol_dir = os.path.join(SOURCE_DIR, pid, 'COBOL')
    if not os.path.exists(cobol_dir):
        print(f"  [SKIP] No COBOL dir for {pid}")
        continue

    cobol_files = sorted([
        f for f in os.listdir(cobol_dir)
        if f.endswith('.cob') or f.endswith('.cbl')
    ])[:1]

    if not cobol_files:
        print(f"  [SKIP] No COBOL files for {pid}")
        continue

    cobol_path = os.path.join(cobol_dir, cobol_files[0])
    print(f"  Translating: {cobol_files[0]}")

    java_outputs, error = translate_cobol_to_java(cobol_path)

    if not java_outputs:
        print(f"  [FAIL] {error}")
        continue

    # Overwrite the pid folder
    pid_output_dir = os.path.join(OUTPUT_DIR, pid)
    if os.path.exists(pid_output_dir):
        shutil.rmtree(pid_output_dir)
    os.makedirs(pid_output_dir)

    # Save ALL java files
    for filename, code in java_outputs.items():
        out_path = os.path.join(pid_output_dir, filename)
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(code)
        print(f"  Saved: {filename}")

    print(f"  {pid}: {len(java_outputs)} Java file(s) saved")

print("\nDone!")



> Running COBOL4J Translations for all 326 programs


In [ ]:
import os
import json
import subprocess
from pathlib import Path
from tqdm import tqdm



# Configuration
SOURCE_DIR = 'CodeNet_Dataset'  
OUTPUT_DIR = 'OUTPUT DIR'
TEMP_DIR = 'temp_cobol'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

# ===== HELPER FUNCTIONS =====

def fix_cobol_format(cobol_code):
    """Fix COBOL formatting for proper column layout"""
    lines = cobol_code.split('\n')
    fixed_lines = []

    for line in lines:
        line = line.rstrip()

        # Skip empty lines
        if not line.strip():
            fixed_lines.append('')
            continue

        # Add 6 spaces for sequence area + 1 space, then code
        fixed_line = '      ' + ' ' + line.lstrip()
        fixed_lines.append(fixed_line)

    return '\n'.join(fixed_lines)


def extract_program_id(fixed_lines):
    """Extract PROGRAM-ID from COBOL code"""
    for line in fixed_lines:
        if 'PROGRAM-ID' in line.upper():
            parts = line.split('.')
            if len(parts) >= 2:
                return parts[1].strip().split()[0]
    return None


def translate_cobol_to_java(cobol_path, temp_dir):
    """
    Translate a single COBOL file to Java

    Returns:
        (java_code, program_id, error, stderr_output)
    """
    try:
        # Read COBOL file
        with open(cobol_path, 'r', encoding='utf-8', errors='ignore') as f:
            cobol_code = f.read()

        # Fix formatting
        fixed_cobol = fix_cobol_format(cobol_code)
        fixed_lines = fixed_cobol.split('\n')

        # Extract PROGRAM-ID
        program_id = extract_program_id(fixed_lines)
        if not program_id:
            # Use filename as fallback
            program_id = Path(cobol_path).stem

        # Save fixed COBOL to temp directory
        temp_cobol_file = os.path.join(temp_dir, f'{program_id}_fixed.cob')
        with open(temp_cobol_file, 'w', encoding='utf-8') as f:
            f.write(fixed_cobol)

        # Run cobj translation
        result = subprocess.run(
            ['cobj', temp_cobol_file],
            capture_output=True,
            text=True,
            timeout=60,
            cwd=temp_dir
        )

        # Check if Java file was generated
        # OpenSourceCOBOL4J generates exactly one .java file, just find it
        java_files = [f for f in os.listdir(temp_dir) if f.endswith('.java')]

        if java_files:
            # There should be only one .java file
            java_filename = java_files[0]
            java_file = os.path.join(temp_dir, java_filename)
            actual_program_id = java_filename.replace('.java', '')

            # Read the Java code
            with open(java_file, 'r', encoding='utf-8') as f:
                java_code = f.read()

            # Clean up temp files
            if os.path.exists(temp_cobol_file):
                os.remove(temp_cobol_file)

            # Remove the java file after reading
            if os.path.exists(java_file):
                os.remove(java_file)

            # Clean up all generated .class files
            for f in os.listdir(temp_dir):
                if f.endswith('.class'):
                    os.remove(os.path.join(temp_dir, f))

            return java_code, actual_program_id, None, result.stderr
        else:
            error_msg = result.stderr if result.stderr else "No Java file generated"
            # Clean up temp COBOL file
            if os.path.exists(temp_cobol_file):
                os.remove(temp_cobol_file)
            return None, program_id, error_msg, result.stderr

    except subprocess.TimeoutExpired:
        return None, program_id, "Translation timeout (>60s)", "Timeout"
    except Exception as e:
        return None, None, str(e), str(e)


# ===== MAIN PROCESSING =====

def process_single_problem(problem_id):
    """Process all COBOL files for a single problem ID"""

    problem_source_dir = os.path.join(SOURCE_DIR, problem_id)
    cobol_dir = os.path.join(problem_source_dir, 'COBOL')

    # Check if COBOL directory exists
    if not os.path.exists(cobol_dir):
        return {
            'problem_id': problem_id,
            'status': 'no_cobol_folder',
            'cobol_count': 0,
            'success': 0,
            'failed': 0
        }

    # Get all COBOL files
    cobol_files = [f for f in os.listdir(cobol_dir)
                   if f.endswith('.cob') or f.endswith('.cbl')]

    if len(cobol_files) == 0:
        return {
            'problem_id': problem_id,
            'status': 'no_cobol_files',
            'cobol_count': 0,
            'success': 0,
            'failed': 0
        }

    # Process only the FIRST COBOL file (alphabetically)
    cobol_files = sorted(cobol_files)[:1]

    # Create output directory for this problem
    problem_output_dir = os.path.join(OUTPUT_DIR, problem_id)
    os.makedirs(problem_output_dir, exist_ok=True)

    # Statistics
    stats = {
        'problem_id': problem_id,
        'status': 'processed',
        'cobol_count': len(cobol_files),
        'success': 0,
        'failed': 0,
        'errors': []
    }

    # Process the COBOL file
    for cobol_filename in cobol_files:
        cobol_path = os.path.join(cobol_dir, cobol_filename)

        # Translate
        java_code, program_id, error, stderr = translate_cobol_to_java(cobol_path, TEMP_DIR)

        if java_code:
            # Save Java file to output directory
            output_filename = f'{program_id}.java'
            output_path = os.path.join(problem_output_dir, output_filename)

            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(java_code)

            stats['success'] += 1
        else:
            stats['failed'] += 1
            stats['errors'].append({
                'file': cobol_filename,
                'error': error,
                'stderr': stderr
            })

    return stats


def main():
    print("="*80)
    print("OpenSourceCOBOL4J BATCH TRANSLATION")
    print("="*80)
    print(f"\nSource: {SOURCE_DIR}")
    print(f"Output: {OUTPUT_DIR}")
    print(f"Mode: Processing 1 COBOL file per problem (first alphabetically)")

    # Get all problem IDs
    all_problems = sorted([
        d for d in os.listdir(SOURCE_DIR)
        if os.path.isdir(os.path.join(SOURCE_DIR, d))
    ])

    print(f"\n Found {len(all_problems)} problems")
    print(f" Starting translation...\n")

    # Process all problems
    all_stats = []

    for problem_id in tqdm(all_problems, desc="Translating"):
        stats = process_single_problem(problem_id)
        all_stats.append(stats)

        # Print progress for this problem
        if stats['status'] == 'processed':
            if stats['success'] > 0:
                tqdm.write(f"   {problem_id}: Success")
            else:
                error_msg = stats['errors'][0]['error'][:60] if stats['errors'] else 'Unknown'
                tqdm.write(f"  {problem_id}: {error_msg}")

    # Save results
    results_file = os.path.join(OUTPUT_DIR, 'translation_results.json')
    with open(results_file, 'w') as f:
        json.dump(all_stats, f, indent=2)

    # Print summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)

    total_cobol = sum(s['cobol_count'] for s in all_stats)
    total_success = sum(s['success'] for s in all_stats)
    total_failed = sum(s['failed'] for s in all_stats)
    problems_with_cobol = sum(1 for s in all_stats if s['status'] == 'processed')

    success_rate = (total_success / total_cobol * 100) if total_cobol > 0 else 0

    print(f"\n Problems processed: {problems_with_cobol}/{len(all_problems)}")
    print(f" Total COBOL files: {total_cobol:,}")
    print(f"\n Success: {total_success:,}/{total_cobol:,} ({success_rate:.1f}%)")
    print(f" Failed:  {total_failed:,}/{total_cobol:,}")

    # Show sample errors if any
    errors_found = [s for s in all_stats if s.get('errors')]
    if errors_found:
        print(f"\n Errors occurred in {len(errors_found)} problems")
        print("\nSample errors (first 5):")
        for stat in errors_found[:5]:
            if stat['errors']:
                error_info = stat['errors'][0]
                print(f"\n  Problem: {stat['problem_id']}")
                print(f"  File: {error_info['file']}")
                print(f"  Error: {error_info['error'][:100]}")

    print(f"\n Results saved to: {results_file}")
    print(f" Translated files in: {OUTPUT_DIR}")

    # Cleanup temp directory
    if os.path.exists(TEMP_DIR):
        import shutil
        shutil.rmtree(TEMP_DIR)

    print("\n Complete!")


if __name__ == "__main__":
    main()

# XMainFrame



> Running translation for all 326 problems using XMainframe.



In [ ]:
import os
import json
import torch
from pathlib import Path
from tqdm import tqdm

# Install dependencies
print(" Installing dependencies...")
os.system("pip install -q bitsandbytes accelerate transformers")

# Configuration
SOURCE_DIR = 'CodeNet_Dataset'
OUTPUT_DIR = 'OUTPUT DIR'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== SETUP XMAINFRAME MODEL =====

print("\n Loading XMainframe model...")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization config (optimized for T4 GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("Fsoft-AIC/XMAiNframe-instruct-7b")

model = AutoModelForCausalLM.from_pretrained(
    "Fsoft-AIC/XMAiNframe-instruct-7b",
    quantization_config=bnb_config,
    device_map="auto"
)

# Verify GPU usage
print(" Model loaded successfully!")
print(f" Model device: {model.device}")
print(f" GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU name: {torch.cuda.get_device_name(0)}")
print()

# ===== TRANSLATION FUNCTION (OPTIMIZED) =====

def translate_with_xmainframe(cobol_code, cobol_filename):
    """
    Translate COBOL to Java using XMainframe with modernization focus

    Args:
        cobol_code: String containing COBOL source code
        cobol_filename: Original filename for reference

    Returns:
        java_code: String containing translated Java code (or None if failed)
        error: Error message if translation failed (or None if success)
    """
    try:

        prompt = f"""[Human]: You are an expert software engineer specializing in legacy code modernization. Translate the following COBOL program to modern, production-ready Java code.

Requirements:

CORRECTNESS:
- Preserve the exact program logic and behavior
- Maintain the same input/output behavior

CODE QUALITY:
- Use modern Java 17 features
- Add brief comments for complex logic

Output ONLY the Java code - no explanations, no markdown code blocks.

COBOL program:

{cobol_code}

[Assistant]:"""

        # Tokenize and move inputs to model device
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # Generate Java code (OPTIMIZED: reduced tokens, removed unnecessary params)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=1024,         # Reduced from 2048 for speed
                do_sample=False,             # Deterministic
                repetition_penalty=1.1,      # Avoid repetitive output
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id
            )

        # Remove the prompt and decode
        generated_tokens = output[0][inputs["input_ids"].shape[1]:]
        java_code = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        # Clean up the output
        java_code = java_code.strip()

        # Remove markdown code blocks if present
        if java_code.startswith("```java"):
            java_code = java_code[7:]
        if java_code.startswith("```"):
            java_code = java_code[3:]
        if java_code.endswith("```"):
            java_code = java_code[:-3]
        java_code = java_code.strip()

        # Check if we got valid output
        if java_code and len(java_code) > 0:
            return java_code, None
        else:
            return None, "Model returned empty or very short output"

    except Exception as e:
        return None, str(e)
    # Removed finally block - no need to clear cache every time


# ===== PIPELINE =====

def process_single_problem(problem_id):
    """Process one COBOL file for a single problem ID"""

    problem_source_dir = os.path.join(SOURCE_DIR, problem_id)
    cobol_dir = os.path.join(problem_source_dir, 'COBOL')

    # Check if COBOL directory exists
    if not os.path.exists(cobol_dir):
        return {
            'problem_id': problem_id,
            'status': 'no_cobol_folder',
            'cobol_count': 0,
            'success': 0,
            'failed': 0
        }

    # Get all COBOL files
    cobol_files = [f for f in os.listdir(cobol_dir)
                   if f.endswith('.cob') or f.endswith('.cbl')]

    if len(cobol_files) == 0:
        return {
            'problem_id': problem_id,
            'status': 'no_cobol_files',
            'cobol_count': 0,
            'success': 0,
            'failed': 0
        }

    # Process only the FIRST COBOL file (alphabetically)
    cobol_files = sorted(cobol_files)[:1]

    # Create output directory for this problem
    problem_output_dir = os.path.join(OUTPUT_DIR, problem_id)
    os.makedirs(problem_output_dir, exist_ok=True)

    # Statistics
    stats = {
        'problem_id': problem_id,
        'status': 'processed',
        'cobol_count': len(cobol_files),
        'success': 0,
        'failed': 0,
        'errors': []
    }

    # Process the COBOL file
    for cobol_filename in cobol_files:
        cobol_path = os.path.join(cobol_dir, cobol_filename)
        base_name = cobol_filename.replace('.cob', '').replace('.cbl', '')

        # Read COBOL source code
        try:
            with open(cobol_path, 'r', encoding='utf-8', errors='ignore') as f:
                cobol_code = f.read()
        except Exception as e:
            stats['failed'] += 1
            stats['errors'].append({
                'file': cobol_filename,
                'error': f"Read error: {str(e)}"
            })
            continue

        # Translate
        java_code, error = translate_with_xmainframe(cobol_code, cobol_filename)

        if java_code:
            # Save translated Java
            output_path = os.path.join(problem_output_dir, f'{base_name}.java')
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(java_code)
            stats['success'] += 1
        else:
            stats['failed'] += 1
            stats['errors'].append({
                'file': cobol_filename,
                'error': error
            })

    return stats


def main():
    print("="*80)
    print("XMainframe COBOL TO JAVA MODERNIZATION (NEW PROMPT)")
    print("="*80)
    print(f"\nSource: {SOURCE_DIR}")
    print(f"Output: {OUTPUT_DIR}")
    print(f"Mode: Processing 1 COBOL file per problem (first alphabetically)")
    print(f"Focus: Production-ready, maintainable Java code")

    # Get all problem IDs
    all_problems = sorted([
        d for d in os.listdir(SOURCE_DIR)
        if os.path.isdir(os.path.join(SOURCE_DIR, d))
    ])

    print(f"\n Processing {len(all_problems)} problems")
    print(f" Starting translation...\n")

    # Process all problems
    all_stats = []

    for problem_id in tqdm(all_problems, desc="Translating"):
        stats = process_single_problem(problem_id)
        all_stats.append(stats)

        # Print progress for this problem
        if stats['status'] == 'processed':
            if stats['success'] > 0:
                tqdm.write(f"  {problem_id}: Success")
            else:
                error_msg = stats['errors'][0]['error'][:60] if stats['errors'] else 'Unknown'
                tqdm.write(f"   {problem_id}: {error_msg}")

    # Save results
    results_file = os.path.join(OUTPUT_DIR, 'translation_results.json')
    with open(results_file, 'w') as f:
        json.dump(all_stats, f, indent=2)

    # Print summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)

    total_cobol = sum(s['cobol_count'] for s in all_stats)
    total_success = sum(s['success'] for s in all_stats)
    total_failed = sum(s['failed'] for s in all_stats)
    problems_with_cobol = sum(1 for s in all_stats if s['status'] == 'processed')

    success_rate = (total_success / total_cobol * 100) if total_cobol > 0 else 0

    print(f"\n Problems processed: {problems_with_cobol}/{len(all_problems)}")
    print(f" Total COBOL files: {total_cobol:,}")
    print(f"\n Success: {total_success:,}/{total_cobol:,} ({success_rate:.1f}%)")
    print(f" Failed:  {total_failed:,}/{total_cobol:,}")

    # Show sample errors if any
    errors_found = [s for s in all_stats if s.get('errors')]
    if errors_found:
        print(f"\n Errors occurred in {len(errors_found)} problems")
        print("\nSample errors (first 5):")
        for stat in errors_found[:5]:
            if stat['errors']:
                error_info = stat['errors'][0]
                print(f"\n  Problem: {stat['problem_id']}")
                print(f"  File: {error_info['file']}")
                print(f"  Error: {error_info['error'][:100]}")

    print(f"\n Results saved to: {results_file}")
    print(f" Translated files in: {OUTPUT_DIR}")
    print("\n Complete!")


if __name__ == "__main__":
    main()

# Qwen




>Running the translation of all 326 problems on Qwen7B



In [ ]:
import os
import json
import torch
from pathlib import Path
from tqdm import tqdm

# Configuration
SOURCE_DIR = 'CodeNet_Dataset'
OUTPUT_DIR = 'OUTPUT DIR'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("SETTING UP QWEN 2.5 CODER MODEL")
print("="*80)

print("\n Installing dependencies...")
os.system("pip install -q transformers accelerate bitsandbytes")

print("\n Loading Qwen model...")

from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_8bit=True
)

print(" Model loaded successfully!")
print(f" Model device: {model.device}")
print(f" GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU name: {torch.cuda.get_device_name(0)}")

def translate_with_qwen(cobol_code, cobol_filename):
    try:
        messages = [
            {
                "role": "user",
                "content": f"""You are an expert software engineer specializing in legacy code modernization. Translate the following COBOL program to modern Java code.

Requirements:

CORRECTNESS:
- Preserve the exact program logic and behavior
- Maintain the same input/output behavior

CODE QUALITY:
- Use modern Java 17 features
- Add brief comments for complex logic

Output ONLY the Java code - no explanations, no markdown code blocks.

COBOL program:

{cobol_code}"""
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                repetition_penalty=1.1,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
            )

        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        java_code = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        if java_code.startswith("```java"):
            java_code = java_code[7:]
        if java_code.startswith("```"):
            java_code = java_code[3:]
        if java_code.endswith("```"):
            java_code = java_code[:-3]
        java_code = java_code.strip()

        if java_code and len(java_code) > 0:
            return java_code, None
        else:
            return None, "Model returned empty or very short output"

    except Exception as e:
        return None, str(e)


def process_single_problem(problem_id):
    problem_source_dir = os.path.join(SOURCE_DIR, problem_id)
    cobol_dir = os.path.join(problem_source_dir, 'COBOL')

    if not os.path.exists(cobol_dir):
        return {'problem_id': problem_id, 'status': 'no_cobol_folder', 'cobol_count': 0, 'success': 0, 'failed': 0}

    cobol_files = sorted([f for f in os.listdir(cobol_dir) if f.endswith('.cob') or f.endswith('.cbl')])[:1]

    if not cobol_files:
        return {'problem_id': problem_id, 'status': 'no_cobol_files', 'cobol_count': 0, 'success': 0, 'failed': 0}

    problem_output_dir = os.path.join(OUTPUT_DIR, problem_id)
    os.makedirs(problem_output_dir, exist_ok=True)

    stats = {'problem_id': problem_id, 'status': 'processed', 'cobol_count': len(cobol_files), 'success': 0, 'failed': 0, 'errors': []}

    for cobol_filename in cobol_files:
        cobol_path = os.path.join(cobol_dir, cobol_filename)
        base_name = cobol_filename.replace('.cob', '').replace('.cbl', '')

        try:
            with open(cobol_path, 'r', encoding='utf-8', errors='ignore') as f:
                cobol_code = f.read()
        except Exception as e:
            stats['failed'] += 1
            stats['errors'].append({'file': cobol_filename, 'error': f"Read error: {str(e)}"})
            continue

        java_code, error = translate_with_qwen(cobol_code, cobol_filename)

        if java_code:
            with open(os.path.join(problem_output_dir, f'{base_name}.java'), 'w', encoding='utf-8') as f:
                f.write(java_code)
            stats['success'] += 1
        else:
            stats['failed'] += 1
            stats['errors'].append({'file': cobol_filename, 'error': error})

    return stats


def main():
    print("\n" + "="*80)
    print("QWEN 2.5 CODER - COBOL TO JAVA TRANSLATION")
    print("="*80)
    print(f"\nSource: {SOURCE_DIR}")
    print(f"Output: {OUTPUT_DIR}")

    all_problems = sorted([
        d for d in os.listdir(SOURCE_DIR)
        if os.path.isdir(os.path.join(SOURCE_DIR, d))
    ])

    print(f"\n Total problems: {len(all_problems)}")
    print(f" Starting translation...\n")

    all_stats = []

    for problem_id in tqdm(all_problems, desc="Translating"):
        stats = process_single_problem(problem_id)
        all_stats.append(stats)

        if stats['status'] == 'processed':
            if stats['success'] > 0:
                tqdm.write(f"  {problem_id}: Success")
            else:
                error_msg = stats['errors'][0]['error'][:60] if stats['errors'] else 'Unknown'
                tqdm.write(f"  {problem_id}: {error_msg}")

    results_file = os.path.join(OUTPUT_DIR, 'translation_results.json')
    with open(results_file, 'w') as f:
        json.dump(all_stats, f, indent=2)

    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)

    total_cobol = sum(s['cobol_count'] for s in all_stats)
    total_success = sum(s['success'] for s in all_stats)
    total_failed = sum(s['failed'] for s in all_stats)
    problems_with_cobol = sum(1 for s in all_stats if s['status'] == 'processed')
    success_rate = (total_success / total_cobol * 100) if total_cobol > 0 else 0

    print(f"\n Problems processed: {problems_with_cobol}/{len(all_problems)}")
    print(f" Total COBOL files: {total_cobol:,}")
    print(f"\n Success: {total_success:,}/{total_cobol:,} ({success_rate:.1f}%)")
    print(f" Failed:  {total_failed:,}/{total_cobol:,}")

    errors_found = [s for s in all_stats if s.get('errors')]
    if errors_found:
        print(f"\n Errors occurred in {len(errors_found)} problems")
        print("\nSample errors (first 5):")
        for stat in errors_found[:5]:
            if stat['errors']:
                print(f"\n  Problem: {stat['problem_id']}")
                print(f"  File: {stat['errors'][0]['file']}")
                print(f"  Error: {stat['errors'][0]['error'][:100]}")

    print(f"\n Results saved to: {results_file}")
    print("\n Complete!")


if __name__ == "__main__":
    main()



> Detecting the version of Java produced by each translator.



In [ ]:
import os
import re
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict


# ===== CONFIGURATION =====
# Add all your translation directories here
TRANSLATION_DIRS = {
    #'qwen': 'Dissertation/Functional/Qwen_results',
    #'xmainframer': 'Dissertation/Functional/XMainframe_results',  # UPDATE THIS PATH
    'cobol4j': 'Dissertation/Functional/Cobol4J_results',  # UPDATE THIS PATH
}


# ===== JAVA FEATURE PATTERNS =====

# Java 8 features (for reference - these are OK)
JAVA8_FEATURES = [
    (r'->', 'Lambda expressions (Java 8)'),
    (r'\.stream\(\)', 'Streams (Java 8)'),
    (r'::' , 'Method references (Java 8)'),
    (r'Optional<', 'Optional (Java 8)'),
    (r'@FunctionalInterface', 'Functional Interface (Java 8)'),
    (r'default\s+\w+.*\{', 'Default methods (Java 8)'),
]

# Java 9+ features (these indicate modern Java)
JAVA9_PLUS_FEATURES = [
    # Java 9
    (r'List\.of\(', 'List.of() (Java 9)'),
    (r'Set\.of\(', 'Set.of() (Java 9)'),
    (r'Map\.of\(', 'Map.of() (Java 9)'),
    (r'Map\.ofEntries\(', 'Map.ofEntries() (Java 9)'),
    (r'Optional\.stream\(', 'Optional.stream() (Java 9)'),
    (r'Stream\.ofNullable\(', 'Stream.ofNullable() (Java 9)'),
    (r'\.takeWhile\(', 'takeWhile() (Java 9)'),
    (r'\.dropWhile\(', 'dropWhile() (Java 9)'),
    (r'@Deprecated\(.*since.*\)', 'Enhanced @Deprecated (Java 9)'),
    (r'module\s+[\w.]+\s*\{', 'Module declaration (Java 9)'),

    # Java 10
    (r'\bvar\s+\w+\s*=', 'var keyword (Java 10)'),

    # Java 11
    (r'\.isBlank\(\)', 'String.isBlank() (Java 11)'),
    (r'\.strip\(\)', 'String.strip() (Java 11)'),
    (r'\.stripLeading\(\)', 'String.stripLeading() (Java 11)'),
    (r'\.stripTrailing\(\)', 'String.stripTrailing() (Java 11)'),
    (r'\.lines\(\)', 'String.lines() (Java 11)'),
    (r'\.repeat\(', 'String.repeat() (Java 11)'),
    (r'Files\.readString\(', 'Files.readString() (Java 11)'),
    (r'Files\.writeString\(', 'Files.writeString() (Java 11)'),
    (r'Predicate\.not\(', 'Predicate.not() (Java 11)'),
    (r'\.toArray\(\w+\[\]::', 'Collection.toArray(IntFunction) (Java 11)'),

    # Java 12
    (r'\.indent\(', 'String.indent() (Java 12)'),
    (r'\.transform\(', 'String.transform() (Java 12)'),
    (r'Files\.mismatch\(', 'Files.mismatch() (Java 12)'),
    (r'switch.*->.*\{', 'Switch expressions (Java 12+)'),

    # Java 13
    (r'"""', 'Text blocks (Java 13-14)'),

    # Java 14
    (r'instanceof\s+\w+\s+\w+\s+&&', 'Pattern matching instanceof (Java 14+)'),
    (r'case\s+.*->', 'Enhanced switch (Java 14)'),
    (r'\byield\s+', 'yield keyword (Java 14)'),

    # Java 15
    (r'\bsealed\s+class', 'Sealed classes (Java 15)'),
    (r'\bsealed\s+interface', 'Sealed interfaces (Java 15)'),
    (r'\bpermits\s+', 'permits clause (Java 15)'),

    # Java 16
    (r'\brecord\s+\w+', 'Records (Java 16)'),
    (r'Stream\.toList\(\)', 'Stream.toList() (Java 16)'),
    (r'instanceof\s+\w+\s+\w+\)', 'Pattern matching for instanceof (Java 16)'),

    # Java 17
    (r'RandomGenerator', 'RandomGenerator (Java 17)'),
    (r'\bsealed\s+.*\bpermits\b', 'Sealed classes (Java 17)'),

    # Java 21 (just in case)
    (r'String\s+\w+\s*=\s*"""', 'String templates (Java 21)'),
    (r'switch.*case.*when', 'Pattern matching switch (Java 21)'),
]


def scan_java_file(file_path):
    """
    Scan a single Java file for modern Java features

    Returns:
        dict with detected features
    """
    result = {
        'java8_features': [],
        'java9plus_features': [],
    }

    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            lines = content.split('\n')

        # Check for Java 8 features
        for pattern, description in JAVA8_FEATURES:
            if re.search(pattern, content):
                result['java8_features'].append(description)

        # Check for Java 9+ features with line numbers
        for pattern, description in JAVA9_PLUS_FEATURES:
            matches = []
            for line_num, line in enumerate(lines, 1):
                if re.search(pattern, line):
                    matches.append({
                        'line': line_num,
                        'code': line.strip()[:100]
                    })

            if matches:
                result['java9plus_features'].append({
                    'feature': description,
                    'occurrences': matches
                })

        return result

    except Exception as e:
        return result


def analyze_directory(source_name, source_dir):
    """
    Analyze all Java files in a directory
    """
    print(f"\n{'='*80}")
    print(f"ANALYZING: {source_name.upper()}")
    print(f"Directory: {source_dir}")
    print(f"{'='*80}\n")

    if not os.path.exists(source_dir):
        print(f" Directory not found!")
        return None

    # Get all problem directories
    problem_dirs = sorted([
        d for d in os.listdir(source_dir)
        if os.path.isdir(os.path.join(source_dir, d))
    ])

    if not problem_dirs:
        print(f"  No subdirectories found!")
        return None

    print(f" Found {len(problem_dirs)} problem directories\n")

    stats = {
        'total_files': 0,
        'files_with_java8': 0,
        'files_with_java9plus': 0,
        'pure_java8_files': 0,
    }

    # Track feature usage
    java8_feature_count = defaultdict(int)
    java9plus_feature_count = defaultdict(int)

    # Sample files with modern features
    modern_feature_examples = []

    for problem_id in tqdm(problem_dirs, desc=f"Scanning {source_name}"):
        problem_dir = os.path.join(source_dir, problem_id)

        # Get all Java files
        java_files = [f for f in os.listdir(problem_dir) if f.endswith('.java')]

        for java_file in java_files:
            stats['total_files'] += 1
            java_path = os.path.join(problem_dir, java_file)

            # Scan the file
            result = scan_java_file(java_path)

            # Update statistics
            has_java8 = len(result['java8_features']) > 0
            has_java9plus = len(result['java9plus_features']) > 0

            if has_java8:
                stats['files_with_java8'] += 1
                for feature in result['java8_features']:
                    java8_feature_count[feature] += 1

            if has_java9plus:
                stats['files_with_java9plus'] += 1
                for feature_info in result['java9plus_features']:
                    java9plus_feature_count[feature_info['feature']] += 1

                # Save example
                if len(modern_feature_examples) < 10:
                    modern_feature_examples.append({
                        'problem_id': problem_id,
                        'file': java_file,
                        'features': result['java9plus_features']
                    })

            # Pure Java 8 = has Java 8 features but no Java 9+ features
            # OR has no modern features at all (basic Java)
            if not has_java9plus:
                stats['pure_java8_files'] += 1

    return {
        'stats': stats,
        'java8_features': dict(java8_feature_count),
        'java9plus_features': dict(java9plus_feature_count),
        'examples': modern_feature_examples
    }


def print_results(source_name, results):
    """
    Print analysis results for a source
    """
    if not results:
        return

    stats = results['stats']
    total = stats['total_files']

    print(f"\n{'='*80}")
    print(f"RESULTS: {source_name.upper()}")
    print(f"{'='*80}\n")

    print(f"📄 Total Java files: {total}")
    print(f"\n Pure Java 8 compatible: {stats['pure_java8_files']}/{total} ({stats['pure_java8_files']/total*100:.1f}%)")
    print(f"Uses Java 9+ features: {stats['files_with_java9plus']}/{total} ({stats['files_with_java9plus']/total*100:.1f}%)")

    # Java 8 features found
    if results['java8_features']:
        print(f"\n Java 8 Features Detected:")
        for feature, count in sorted(results['java8_features'].items(), key=lambda x: x[1], reverse=True):
            print(f"   {feature}: {count} files")

    # Java 9+ features found
    if results['java9plus_features']:
        print(f"\n Java 9+ Features Detected:")
        for feature, count in sorted(results['java9plus_features'].items(), key=lambda x: x[1], reverse=True)[:15]:
            print(f"   {feature}: {count} occurrences")
    else:
        print(f"\n No Java 9+ features detected - all code is Java 8 compatible!")

    # Show examples
    if results['examples']:
        print(f"\n Sample Files with Java 9+ Features (showing first 5):")
        for i, example in enumerate(results['examples'][:5], 1):
            print(f"\n{i}. {example['problem_id']}/{example['file']}:")
            for feature_info in example['features'][:3]:  # Show first 3 features per file
                print(f"   • {feature_info['feature']}")
                if feature_info['occurrences']:
                    occ = feature_info['occurrences'][0]
                    print(f"     Line {occ['line']}: {occ['code']}")


def compare_sources(all_results):
    """
    Print comparison table across all sources
    """
    print(f"\n{'='*80}")
    print("CROSS-SOURCE COMPARISON")
    print(f"{'='*80}\n")

    # Summary table
    print(f"{'Source':<20} {'Total Files':<15} {'Java 8 Only':<20} {'Has Java 9+':<20}")
    print("-" * 75)

    for source_name, results in all_results.items():
        if not results:
            continue
        stats = results['stats']
        total = stats['total_files']
        if total > 0:
            java8_pct = f"{stats['pure_java8_files']}/{total} ({stats['pure_java8_files']/total*100:.1f}%)"
            java9_pct = f"{stats['files_with_java9plus']}/{total} ({stats['files_with_java9plus']/total*100:.1f}%)"
        else:
            java8_pct = "N/A"
            java9_pct = "N/A"

        print(f"{source_name:<20} {total:<15} {java8_pct:<20} {java9_pct:<20}")

    # Feature comparison
    print(f"\n{'='*80}")
    print("JAVA 9+ FEATURE USAGE BY SOURCE")
    print(f"{'='*80}\n")

    # Get all unique features
    all_features = set()
    for results in all_results.values():
        if results and results['java9plus_features']:
            all_features.update(results['java9plus_features'].keys())

    if all_features:
        # Print header
        print(f"{'Feature':<50}", end="")
        for source_name in all_results.keys():
            print(f"{source_name:<15}", end="")
        print()
        print("-" * (50 + 15 * len(all_results)))

        # Print each feature
        for feature in sorted(all_features):
            print(f"{feature:<50}", end="")
            for source_name, results in all_results.items():
                if results and results['java9plus_features']:
                    count = results['java9plus_features'].get(feature, 0)
                else:
                    count = 0
                print(f"{count:<15}", end="")
            print()
    else:
        print(" No Java 9+ features found in any source!")

    # Verdict
    print(f"\n{'='*80}")
    print("VERDICT")
    print(f"{'='*80}\n")

    for source_name, results in all_results.items():
        if not results:
            continue
        stats = results['stats']
        if stats['files_with_java9plus'] == 0:
            print(f" {source_name}: Pure Java 8 - No modern features detected")
        else:
            pct = stats['files_with_java9plus'] / stats['total_files'] * 100
            print(f" {source_name}: Uses Java 9+ features in {pct:.1f}% of files")


def main():
    """
    Main function to analyze all translation sources
    """
    print("="*80)
    print("JAVA VERSION FEATURE ANALYZER")
    print("Detecting Java 8 vs Java 9+ features without compilation")
    print("="*80)

    all_results = {}

    for source_name, source_dir in TRANSLATION_DIRS.items():
        results = analyze_directory(source_name, source_dir)
        all_results[source_name] = results
        if results:
            print_results(source_name, results)

    # Print comparison if we have multiple sources
    valid_results = {k: v for k, v in all_results.items() if v is not None}
    if len(valid_results) > 1:
        compare_sources(valid_results)

    print("\n" + "="*80)
    print(" Analysis complete!")
    print("="*80)


if __name__ == "__main__":
    main()



> Cleaning the LLM-generated java code - removing backticks, extra text etc.



In [ ]:
"""
Clean LLM Java files - remove LLM commentary from top and bottom
Process all PIDs and save to xmainframe_final
"""

import os
import re
from pathlib import Path
import shutil

def clean_llm_commentary(content):
    """
    Remove LLM commentary from both the start and end of Java code.
    Finds where actual Java code begins and ends, removing everything outside.
    """
    lines = content.split('\n')

    # Find the first line that looks like actual Java code
    java_start_patterns = [
        r'^\s*package\s+',
        r'^\s*import\s+',
        r'^\s*public\s+class\s+',
        r'^\s*class\s+',
        r'^\s*public\s+interface\s+',
        r'^\s*/\*\*',  # Javadoc comment
    ]

    llm_indicators = [
        '<|assistant|>',
        'here\'s the translated',
        'here is the',
        'this cobol',
        'the task is',
        'the given cobol',
        '```java',
        '```',
        'explanation:',
        '### explanation',
        '## explanation',
    ]

    # Find start of actual Java code
    start_idx = 0
    for i, line in enumerate(lines):
        stripped = line.strip().lower()

        # Skip LLM commentary lines
        if any(indicator in stripped for indicator in llm_indicators):
            continue

        # Check if this line matches any Java code pattern
        for pattern in java_start_patterns:
            if re.match(pattern, line):
                start_idx = i
                break

        if start_idx > 0:
            break

        # Also check for common Java code indicators
        if line.strip() and not any(indicator in stripped for indicator in llm_indicators):
            # Check if it looks like code (has semicolon, braces, etc.)
            if any(char in line for char in ['{', '}', ';']) and 'translation' not in stripped:
                start_idx = i
                break

    # Find end of actual Java code (last closing brace of main class)
    # Work backwards from the end
    end_idx = len(lines)

    for i in range(len(lines) - 1, -1, -1):
        stripped = lines[i].strip().lower()

        # Skip LLM commentary at the end
        if any(indicator in stripped for indicator in llm_indicators):
            continue

        # Skip markdown backticks
        if stripped.startswith('```'):
            continue

        # Find last closing brace (likely end of class)
        if '}' in lines[i]:
            end_idx = i + 1
            break

    # Extract the Java code portion
    if start_idx < end_idx:
        return '\n'.join(lines[start_idx:end_idx])

    # If we couldn't find boundaries, return original
    return content

def remove_markdown_backticks(content):
    """Remove markdown code fences if present"""
    # Remove opening ```java or ```
    content = re.sub(r'^```(?:java)?\s*\n', '', content, flags=re.MULTILINE)

    # Remove closing ```
    content = re.sub(r'\n```\s*$', '', content, flags=re.MULTILINE)

    # Remove any ``` in the middle
    lines = content.split('\n')
    cleaned_lines = [line for line in lines if not line.strip().startswith('```')]

    return '\n'.join(cleaned_lines)

def process_java_file(filepath):
    """
    Process a Java file - clean LLM commentary from top and bottom
    """
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

        # Remove markdown backticks first
        content = remove_markdown_backticks(content)

        # Remove LLM commentary from top and bottom
        content = clean_llm_commentary(content)

        # Clean up extra whitespace at start/end
        content = content.strip()

        # Add newline at end of file (Java convention)
        if content and not content.endswith('\n'):
            content += '\n'

        return content
    except Exception as e:
        print(f"   Error reading file: {e}")
        return None

def clean_all_files(source_dir, target_dir):
    """
    Clean all Java files from source and save to target
    """
    source_path = Path(source_dir)
    target_path = Path(target_dir)

    if not source_path.exists():
        print(f" Source directory not found: {source_dir}")
        return

    # Create target directory
    target_path.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*80}")
    print(f"Cleaning ALL XMainframe Java Files")
    print(f"Source: {source_dir}")
    print(f"Target: {target_dir}")
    print(f"{'='*80}\n")

    total_files = 0
    cleaned_files = 0
    errors = 0
    skipped_pids = []

    # Get all PID folders
    pid_folders = sorted([d for d in source_path.iterdir() if d.is_dir()])

    print(f"Found {len(pid_folders)} PID folders to process\n")

    # Process each PID folder
    for pid_folder in pid_folders:
        pid = pid_folder.name

        # Create corresponding folder in target
        target_pid_folder = target_path / pid
        target_pid_folder.mkdir(parents=True, exist_ok=True)

        # Get all .java files in this PID
        java_files = list(pid_folder.glob("*.java"))

        if not java_files:
            skipped_pids.append(pid)
            continue

        # Process each .java file
        for java_file in java_files:
            total_files += 1

            # Clean the file
            content = process_java_file(java_file)

            if content is not None:
                # Write to target location
                target_file = target_pid_folder / java_file.name

                try:
                    with open(target_file, 'w', encoding='utf-8') as f:
                        f.write(content)

                    cleaned_files += 1

                    # Print progress every 50 files
                    if cleaned_files % 50 == 0:
                        print(f" Cleaned {cleaned_files} files...")

                except Exception as e:
                    print(f" Error writing {pid}/{java_file.name}: {e}")
                    errors += 1
            else:
                errors += 1

    # Print summary
    print(f"\n{'='*80}")
    print(f"Processing Complete!")
    print(f"{'='*80}")
    print(f"Total files processed: {total_files}")
    print(f"Successfully cleaned: {cleaned_files}")
    print(f"Errors: {errors}")
    print(f"PIDs with no Java files: {len(skipped_pids)}")
    print(f"\nOutput directory: {target_dir}")
    print(f"{'='*80}\n")

    if errors > 0:
        print(f" {errors} files had errors during processing")

    if skipped_pids:
        print(f"\nPIDs with no Java files (first 10): {skipped_pids[:10]}")

if __name__ == "__main__":
    # Mount Google Drive (for Colab)

    source_dir = "PATH TO TRANSLATION BEFORE CLEANING (e.g. XMainframe_results)"
    target_dir = "PATH TO FINAL OUTPUT DIRECTORY (e.g. XMainframe_results_final)"

    # Process all files
    clean_all_files(source_dir, target_dir)

    print("\n All files have been cleaned and saved to xmainframe_final!")